# 基礎設定

In [ ]:
import sqlite3
conn = sqlite3.connect('stocks.db') # 如果資料庫不存在，會自動幫你建立
sql_create_table = """
CREATE TABLE `stock_date` (
	`stock_id`	TEXT,
	`date`	TEXT,
	`open`	REAL,
	`high`	REAL,
	`low`	REAL,
	`close`	REAL,
	`volume`	INTEGER
)
"""
cursor = conn.execute(sql_create_table)
conn.close()

In [ ]:
NGROK_AUTHTOKEN = "YOUR_NGROK_AUTHTOKEN"

In [ ]:
!pip install pyngrok flask --quiet

# SQL

## SQLite

### 建立資料表 (Table)

#### [練習] 建立 `stock_list` 資料表
寫出建立資料表 `stock_list` 的 SQLite 語法
- 股票代號 `stock_id`
    - 例如 NVDA
- 股票名稱 `name`
    - 例如 NVIDIA

In [ ]:
import sqlite3
conn = sqlite3.connect('stocks.db')
sql_create_table = """
CREATE TABLE `stock_list` (
	`stock_id`	TEXT,
	`name`	TEXT
)
"""
cursor = conn.execute(sql_create_table)
conn.close()

## CRUD

### Create 新增資料

#### [練習] 新增資料到 `stock_list`
剛才我們建立了資料表 `stock_list`，有兩個欄位
- 股票代號 `stock_id`
- 股票名稱 `name`

但還沒有把資料加入，請將以下股票名稱、股票代號加入資料庫：

    
| 股票代號 | 股票名稱 |
| --- | --- |
| NVDA | NVIDIA |
| AAPL | APPLE |
| GOOGL | GOOGLE |


In [ ]:
import sqlite3
conn = sqlite3.connect('stocks.db')
sql_ins = """
    INSERT INTO `stock_list` (`stock_id`, `name`)
    VALUES ( 'NVDA', 'NVIDIA' )
"""
cursor = conn.execute(sql_ins)
cursor = conn.commit()

sql_ins = """
    INSERT INTO `stock_list` (`stock_id`, `name`)
    VALUES ( 'AAPL', 'APPLE' )
"""
cursor = conn.execute(sql_ins)
cursor = conn.commit()

# 處理 GOOGLE
sql_ins = """
    INSERT INTO `stock_list` (`stock_id`, `name`)
    VALUES ( 'GOOGL', 'GOOGLE' )
"""
cursor = conn.execute(sql_ins)
cursor = conn.commit()

conn.close()

In [ ]:
import sqlite3
stock_info = [
    ['NVDA', 'NVIDIA'],
    ['AAPL', 'APPLE'],
    ['GOOGL', 'GOOGLE'],
]

conn = sqlite3.connect('stocks.db')
for stock_id, stock_name in stock_info:
    sql_ins = f"""
        INSERT INTO `stock_list` (`stock_id`, `name`)
        VALUES ( '{stock_id}', '{stock_name}' )
    """
    cursor = conn.execute(sql_ins)
    cursor = conn.commit()
conn.close()

#### [進階練習] 新增資料到 `stock_date`
將以下兩檔股票的資料加入到 `stock_date`

##### 實戰要點
- 原始資料沒有股票代號，要記得一併寫入 (`NVDA` 與 `AAPL`)
- 雖然 SQLite 對型別比較寬容，但養成好習慣：INTEGER 欄位用 `int()`、REAL 欄位用 `float()` 先轉型再寫入
- 可搭配 Python 的 `for` 迴圈與字串格式化批次新增，一次只連線一次、一次 commit

In [ ]:
import sqlite3
datas_NVDA = [
    ['2026-04-20', '199.98', '202.17', '197.84', '202.06', '119381400'],
    ['2026-04-21', '202.13', '202.75', '199', '199.88', '107945302'],
    ['2026-04-22', '200.99', '202.5', '199', '202.5', '107501042'],
    ['2026-04-23', '202.46', '203.83', '197.22', '199.64', '113561830'],
    ['2026-04-24', '199.96', '210.965', '199.81', '208.26', '166563954']
]
datas_AAPL = [
    ['2026-04-20', '270.33', '274.27', '270.29', '273.05', '36590200'],
    ['2026-04-21', '271.5', '272.8', '265.4', '266.17', '50209800'],
    ['2026-04-22', '267.82', '273.74', '266.87', '273.17', '43249204'],
    ['2026-04-23', '275.05', '275.77', '271.65', '273.43', '33399639'],
    ['2026-04-24', '272.76', '273.06', '269.67', '270.11', '20541211']
]
def ins_stock_data(stock_id, datas):
    conn = sqlite3.connect('stocks.db')
    for row in datas:
        d, o, h, l, c, v = row
        sql_ins = """
            INSERT INTO `stock_date` (`stock_id`, `date`, `open`, `high`, `low`, `close`, `volume`)
            VALUES ( '{}' ,'{}', {}, {}, {}, {}, {} )
        """.format(stock_id, d, float(o), float(h), float(l), float(c), int(v))
        cursor = conn.execute(sql_ins)
    conn.commit()
    conn.close()

ins_stock_data('NVDA', datas_NVDA)
ins_stock_data('AAPL', datas_AAPL)

### Read 讀取資料

#### [練習] NVIDIA 股價資料整理與最高價篩選
- 列出 NVIDIA (NVDA) 2026-04-21 ~ 2026-04-24 的每日資訊
    - 日期
    - 開盤、收盤、最高、最低
    - **均價 CDP**：`(最高 + 最低 + 2 * 收盤) / 4`
- 列出擁有最大「最高價」的兩筆資料

> 提示：`ORDER BY` 搭配 `LIMIT` 可以快速取出最大的幾筆。

In [ ]:
import sqlite3
conn = sqlite3.connect('stocks.db')
sql = """
    SELECT `date`, `open`, `close`, `high`, `low`, (`high`+`low`+2*`close`)/4
    FROM `stock_date`
    WHERE `stock_id` = 'NVDA' and `date` >= '2026-04-21' and `date` <= '2026-04-24'
    ORDER BY `date`
"""
cursor = conn.execute(sql)
for row in cursor.fetchall():
    print(row)
conn.close()

('2026-04-21', 202.13, 199.88, 202.75, 199.0, 200.3775)
('2026-04-22', 200.99, 202.5, 202.5, 199.0, 201.625)
('2026-04-23', 202.46, 199.64, 203.83, 197.22, 200.08249999999998)
('2026-04-24', 199.96, 208.26, 210.965, 199.81, 206.82375)


列出擁有最大「最高價」的兩筆資料

In [ ]:
import sqlite3
conn = sqlite3.connect('stocks.db')
sql = """
    SELECT `stock_id`, `date`, `open`, `high`, `low`, `close`, `volume`
    FROM `stock_date`
    ORDER BY `high` DESC
    LIMIT 2
"""
cursor = conn.execute(sql)
for row in cursor.fetchall():
    print(row)
conn.close()

('AAPL', '2026-04-23', 275.05, 275.77, 271.65, 273.43, 33399639)
('AAPL', '2026-04-20', 270.33, 274.27, 270.29, 273.05, 36590200)


# Flask

## 基本 route

### [練習] 股票列表頁
寫出股票列表頁：
- 網址 `/stock_list`
- 表格有三列兩欄（股票代號、股票名）
    
    
| 股票代號 | 股票名稱 |
| --- | --- |
| NVDA | NVIDIA |
| AAPL | APPLE |
| GOOGL | GOOGLE |


In [ ]:
from flask import Flask
from pyngrok import ngrok
ngrok.set_auth_token(NGROK_AUTHTOKEN)
ngrok.kill()

app = Flask(__name__)

@app.route("/")
def home():
    resp = """
        Welcome to Stock Board
        <br>
        <a href='/stock_list'>進入股票列表頁</a>
        <br>
        <a href='/stock'>進入股票頁</a>
    """
    return resp

@app.route("/stock_list") # 修改這裡
def stock_list():
    datas = [
        ['NVDA',  'NVIDIA'],
        ['AAPL',  'APPLE'],
        ['GOOGL', 'GOOGLE'],
    ]
    html_data = ""
    for row in datas:
        stock_id, name = row
        html_row = """
            <tr>
                <td>{}</td>
                <td>{}</td>
            </tr>""".format(stock_id, name) # 修改這裡
        html_data += html_row
    resp = """
        <h1>股票列表</h1>
        <a href='/'>回首頁</a>
        <table>
            <thead>
                <tr>
                    <th>股票代碼</th>
                    <th>股票名</th>
                </tr>
            </thead>
            <tbody>
                {}
            </tbody>
        </table>
    """.format(html_data)
    return resp

@app.route("/stock")
def stock():
    stock_id = "NVDA"
    datas_NVDA = [
        ['2026-04-22', 200.99, 202.5, 199, 202.5, 107501042],
        ['2026-04-23', 202.46, 203.83, 197.22, 199.64, 113561830],
        ['2026-04-24', 199.96, 210.965, 199.81, 208.26, 166563954]
    ]
    html_data = ""
    for row in datas_NVDA:
        d, o, h, l, c, v = row
        html_row = """
            <tr>
                <td>{}</td>
                <td>{}</td>
                <td>{}</td>
                <td>{}</td>
                <td>{}</td>
                <td>{}</td>
            </tr>""".format(d, o, h, l, c, v)
        html_data += html_row
    resp = """
        <h1>股票代碼: {}</h1>
        <a href='/'>回首頁</a>
        <table>
            <thead>
                <tr>
                    <th>日期</th>
                    <th>開盤價</th>
                    <th>最高價</th>
                    <th>最低價</th>
                    <th>收盤價</th>
                    <th>成交量</th>
                </tr>
            </thead>
            <tbody>
                {}
            </tbody>
        </table>
    """.format(stock_id, html_data)
    return resp

public_url = ngrok.connect(5000)
print(" * 公開網址:", public_url)

app.run()

 * 公開網址: NgrokTunnel: "https://b26e-34-91-105-255.ngrok-free.app" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [25/Apr/2026 02:45:24] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [25/Apr/2026 02:45:25] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [25/Apr/2026 02:45:26] "GET /stock_list HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [25/Apr/2026 02:45:27] "GET /stock HTTP/1.1" 200 -


## Template 模板

### [練習] 首頁模板化
將首頁也改成 Template（檔案名稱：`index.html`）
- 把原本 `home()` 裡的 HTML 字串搬進 `templates/index.html`
- `home()` 改成 `return render_template`

In [ ]:
import os

dir_name = "templates"
if dir_name not in os.listdir():
    os.makedirs(dir_name)

index_html = """
Welcome to Stock Board
<br>
<a href='/stock'>進入股票頁</a>
"""
with open("templates/index.html","w") as fo:
    fo.write(index_html)

In [ ]:
from flask import Flask, render_template
from pyngrok import ngrok
ngrok.set_auth_token(NGROK_AUTHTOKEN)
ngrok.kill()

app = Flask(__name__)

@app.route("/")
def home():
    return render_template("index.html")

public_url = ngrok.connect(5000)
print(" * 公開網址:", public_url)

app.run()

 * 公開網址: NgrokTunnel: "https://f2be-34-91-105-255.ngrok-free.app" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [25/Apr/2026 02:50:51] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [25/Apr/2026 02:50:52] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [25/Apr/2026 02:50:54] "GET /stock HTTP/1.1" 404 -


## Parameter 網址參數

### [練習] 增加 `date_start` 參數
- 增加一個參數 `date_start`
    - 如果走 `get_stock_path`，url 設計成 `/stock/NVDA/YYYY-MM-DD`
    - 如果走 `get_stock_param`，url 設計成 `/stock?stock_id=NVDA&date_start=YYYY-MM-DD`
- 先把收到的 `date_start` `print` 出來確認有正確接到

> 提示：  
> path 版的網頁路徑和函式傳入的參數要增加 `date_start`   
> query 版的 `date_start` 則是要用 request.args.get 取得，是在函式內增加一行程式碼。

In [ ]:
dir_name = "templates"
if dir_name not in os.listdir():
    os.makedirs(dir_name)

stock_html = """
<h1>股票代碼: {{ stock_id }}</h1>
<a href='/'>回首頁</a>
<table>
    <thead>
        <tr>
            <th>日期</th>
            <th>開盤價</th>
            <th>最高價</th>
            <th>最低價</th>
            <th>收盤價</th>
            <th>成交量</th>
        </tr>
    </thead>
    <tbody>
        {% for row in datas %}
        <tr>
            <td>{{ row[0] }}</td>
            <td>{{ row[1] }}</td>
            <td>{{ row[2] }}</td>
            <td>{{ row[3] }}</td>
            <td>{{ row[4] }}</td>
            <td>{{ row[5] }}</td>
        </tr>
        {% endfor %}
    </tbody>
</table>
"""
with open("templates/stock.html","w") as fo:
    fo.write(stock_html)

In [ ]:
from flask import Flask, render_template, request
from pyngrok import ngrok
ngrok.set_auth_token(NGROK_AUTHTOKEN)
ngrok.kill()

app = Flask(__name__)

def get_stock_datas(stock_id, date_start):
    print("date_start =", date_start)
    datas = [
        ['2026-04-22', 200.99, 202.5, 199, 202.5, 107501042],
        ['2026-04-23', 202.46, 203.83, 197.22, 199.64, 113561830],
        ['2026-04-24', 199.96, 210.965, 199.81, 208.26, 166563954]
    ]
    return datas

@app.route("/")
def home():
    resp = """Welcome to Stock Board
    <br>
    <a href='/stock/NVDA/2026-04-22'>進入NVDA股票頁(路徑)</a>
    <br>
    <a href='/stock?stock_id=NVDA&date_start=2026-04-22'>進入NVDA股票頁(參數)</a>
    """
    return resp

@app.route("/stock/<stock_id>/<date_start>")
def get_stock_path(stock_id, date_start):
    return render_template("stock.html", stock_id=stock_id, date_start=date_start, datas=get_stock_datas(stock_id, date_start))

@app.route("/stock")
def get_stock_param():
    stock_id = request.args.get('stock_id')
    date_start = request.args.get('date_start')
    return render_template("stock.html", stock_id=stock_id, date_start=date_start, datas=get_stock_datas(stock_id, date_start))

public_url = ngrok.connect(5000)
print(" * 公開網址:", public_url)

app.run()

 * 公開網址: NgrokTunnel: "https://90f3-34-91-105-255.ngrok-free.app" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [25/Apr/2026 03:05:38] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [25/Apr/2026 03:05:38] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [25/Apr/2026 03:05:40] "GET /stock/NVDA/2026-04-22 HTTP/1.1" 200 -


date_start = 2026-04-22


## 串接 SQLite 資料庫

### [練習]
延續上一個練習，把 `date_start` 當作 SQL 查詢條件：搜尋此日期（包含）之後的每日資訊
- 注意日期在 SQLite 是以 TEXT 儲存，WHERE 條件記得用單引號：`` `date` >= 'YYYY-MM-DD' ``
- 想想看：如果 `date_start` 沒傳入（為 `None`）時，SQL 該怎麼組才不會出錯？

In [ ]:
from flask import Flask, render_template, request
from pyngrok import ngrok
import sqlite3
ngrok.set_auth_token(NGROK_AUTHTOKEN)
ngrok.kill()

app = Flask(__name__)

def get_stock_datas(stock_id, date_start):
    conn = sqlite3.connect('stocks.db')
    sql = """
        SELECT `date`, `open`, `high`, `low`, `close`, `volume`
        FROM `stock_date`
        WHERE `stock_id` = '{}' and `date` = '{}'
        ORDER BY `date`
    """.format(stock_id, date_start)
    cursor = conn.execute(sql)
    datas = cursor.fetchall()
    conn.close()
    return datas

@app.route("/")
def home():
    resp = """Welcome to Stock Board
    <br>
    <a href='/stock/NVDA/2026-04-22'>進入NVDA股票頁(路徑)</a>
    <br>
    <a href='/stock?stock_id=NVDA&date_start=2026-04-22'>進入NVDA股票頁(參數)</a>
    """
    return resp

@app.route("/stock/<stock_id>/<date_start>")
def get_stock_path(stock_id, date_start):
    return render_template("stock.html", stock_id=stock_id, date_start=date_start, datas=get_stock_datas(stock_id, date_start))

@app.route("/stock")
def get_stock_param():
    stock_id = request.args.get('stock_id')
    date_start = request.args.get('date_start')
    return render_template("stock.html", stock_id=stock_id, date_start=date_start, datas=get_stock_datas(stock_id, date_start))

public_url = ngrok.connect(5000)
print(" * 公開網址:", public_url)

app.run()

 * 公開網址: NgrokTunnel: "https://2832-34-91-105-255.ngrok-free.app" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [25/Apr/2026 03:10:42] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [25/Apr/2026 03:10:43] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [25/Apr/2026 03:10:44] "GET /stock/NVDA/2026-04-22 HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [25/Apr/2026 03:10:46] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [25/Apr/2026 03:10:47] "GET /stock?stock_id=NVDA&date_start=2026-04-22 HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [25/Apr/2026 03:10:49] "GET / HTTP/1.1" 200 -


In [ ]:
from flask import Flask, render_template, request
from pyngrok import ngrok
import sqlite3
ngrok.set_auth_token(NGROK_AUTHTOKEN)
ngrok.kill()

app = Flask(__name__)

def get_stock_datas(stock_id, date_start):
    conn = sqlite3.connect('stocks.db')
    date_condition = ""
    if date_start:
        date_condition = f"and `date` = '{date_start}'"
    sql = """
        SELECT `date`, `open`, `high`, `low`, `close`, `volume`
        FROM `stock_date`
        WHERE `stock_id` = '{}' {}
        ORDER BY `date`
    """.format(stock_id, date_condition)
    cursor = conn.execute(sql)
    datas = cursor.fetchall()
    conn.close()
    return datas

@app.route("/")
def home():
    resp = """Welcome to Stock Board
    <br>
    <a href='/stock/NVDA/2026-04-22'>進入NVDA股票頁(路徑)</a>
    <br>
    <a href='/stock?stock_id=NVDA&date_start=2026-04-22'>進入NVDA股票頁(參數)</a>
    """
    return resp

@app.route("/stock/<stock_id>/<date_start>")
def get_stock_path(stock_id, date_start):
    return render_template("stock.html", stock_id=stock_id, date_start=date_start, datas=get_stock_datas(stock_id, date_start))

@app.route("/stock")
def get_stock_param():
    stock_id = request.args.get('stock_id')
    date_start = request.args.get('date_start')
    return render_template("stock.html", stock_id=stock_id, date_start=date_start, datas=get_stock_datas(stock_id, date_start))

public_url = ngrok.connect(5000)
print(" * 公開網址:", public_url)

app.run()

 * 公開網址: NgrokTunnel: "https://2c35-34-91-105-255.ngrok-free.app" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [25/Apr/2026 03:11:57] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [25/Apr/2026 03:11:58] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [25/Apr/2026 03:11:59] "GET /stock/NVDA/2026-04-22 HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [25/Apr/2026 03:12:02] "GET /stock/NVDA/ HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [25/Apr/2026 03:12:09] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [25/Apr/2026 03:12:11] "GET /stock?stock_id=NVDA&date_start=2026-04-22 HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [25/Apr/2026 03:12:15] "GET /stock?stock_id=NVDA HTTP/1.1" 200 -
